In [ ]:
print("Hello")

Hello


In [1]:
import torch

print("=" * 50)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

CUDA Available : True
GPU : Tesla T4


In [2]:
!git clone https://github.com/ver1619/rag-bench.git

Cloning into 'rag-bench'...
remote: Enumerating objects: 234, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 234 (delta 10), reused 16 (delta 6), pack-reused 214 (from 1)
Receiving objects: 100% (234/234), 63.88 MiB | 18.15 MiB/s, done.
Resolving deltas: 100% (75/75), done.


In [3]:
%cd rag-bench

/content/rag-bench


In [4]:
!pwd

/content/rag-bench


In [5]:
!ls

data  docker-compose.yml  notebooks  README.md	scripts  src


In [6]:
import sys

print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [8]:
!pip install langchain-text-splitters nltk pymupdf sentence-transformers torch qdrant-client rank-bm25 python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 33.5 MB/s eta 0:00:00


In [9]:
import os

print(os.getcwd())

/content/rag-bench


In [10]:
!pwd
!find . -maxdepth 2 -type d

/content/rag-bench
.
./src
./src/__pycache__
./src/chunkers
./src/ingestion
./src/parser
./src/embeddings
./src/indexing
./src/validators
./src/models
./src/vectordb
./src/rerankers
./src/utils
./src/pipeline
./src/evaluation
./src/settings
./src/retrieval
./.git
./.git/hooks
./.git/objects
./.git/info
./.git/refs
./.git/logs
./.git/branches
./scripts
./scripts/__pycache__
./data
./data/queries
./data/metadata
./data/documents
./notebooks


In [11]:
!find src -maxdepth 2 -type f

src/__pycache__/__init__.cpython-312.pyc
src/chunkers/sentence.py
src/chunkers/fixed.py
src/chunkers/recursive.py
src/chunkers/base.py
src/chunkers/__init__.py
src/ingestion/__init__.py
src/ingestion/service.py
src/parser/__init__.py
src/parser/pdf_parser.py
src/embeddings/sentence_transformer.py
src/embeddings/base.py
src/embeddings/__init__.py
src/indexing/__init__.py
src/indexing/service.py
src/validators/ingestion.py
src/validators/pipeline.py
src/validators/embedding.py
src/validators/base.py
src/validators/__init__.py
src/validators/chunking.py
src/models/vector_point.py
src/models/embedding.py
src/models/chunk.py
src/models/document.py
src/models/__init__.py
src/vectordb/qdrant.py
src/vectordb/base.py
src/vectordb/__init__.py
src/rerankers/cross_encoder.py
src/rerankers/base.py
src/rerankers/__init__.py
src/rerankers/factory.py
src/rerankers/models.py
src/utils/file_utils.py
src/utils/__init__.py
src/pipeline/builder.py
src/pipeline/artifacts.py
src/pipeline/__init__.py
src/pipe

In [12]:
import sys
import os

print("Current Working Directory:", os.getcwd())
print("Python Version:", sys.version)

Current Working Directory: /content/rag-bench
Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [13]:
from src.settings.paths import *
print("✅ settings imported")

✅ settings imported


In [15]:
!pip install -q qdrant-client

In [16]:
from google.colab import userdata
from qdrant_client import QdrantClient

QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

print("✅ Connected successfully!")

✅ Connected successfully!


In [17]:
collections = client.get_collections()

print(collections)

collections=[]


In [18]:
collections = client.get_collections()

print("Collections:\n")

if collections.collections:
    for c in collections.collections:
        print("-", c.name)
else:
    print("No collections found.")

Collections:

No collections found.


In [19]:
from qdrant_client.models import Distance, VectorParams

client.create_collection(
    collection_name="test_connection",
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE,
    ),
)

print("✅ Test collection created.")

✅ Test collection created.


In [20]:
collections = client.get_collections()

for c in collections.collections:
    print(c.name)

test_connection


In [21]:
client.delete_collection("test_connection")

print("✅ Test collection deleted.")

✅ Test collection deleted.


In [22]:
from qdrant_client.models import VectorParams, Distance

COLLECTION = "smoke_test"

client.recreate_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(
        size=4,
        distance=Distance.COSINE
    )
)

print("✅ Collection created")

/tmp/ipykernel_827/3950092840.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


✅ Collection created


In [23]:
from qdrant_client.models import PointStruct

client.upsert(
    collection_name=COLLECTION,
    wait=True,
    points=[
        PointStruct(
            id=1,
            vector=[0.1, 0.2, 0.3, 0.4],
            payload={
                "document": "Test Document",
                "source": "Colab Smoke Test"
            }
        )
    ]
)

print("✅ Vector uploaded")

✅ Vector uploaded


In [24]:
result = client.query_points(
    collection_name=COLLECTION,
    query=[0.1, 0.2, 0.3, 0.4],
    limit=1
)

print(result)

points=[ScoredPoint(id=1, version=1, score=1.0, payload={'document': 'Test Document', 'source': 'Colab Smoke Test'}, vector=None, shard_key=None, order_value=None)]


In [25]:
point = result.points[0]

print("ID:", point.id)
print("Score:", point.score)
print("Payload:", point.payload)

ID: 1
Score: 1.0
Payload: {'document': 'Test Document', 'source': 'Colab Smoke Test'}


In [26]:
client.delete_collection(COLLECTION)

print("✅ Collection deleted")

✅ Collection deleted
